In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data/clean/en_uz/hplt/hplt_en_uz_10k_clean.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results/data_quality/hplt_en_uz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_PATH = OUTPUT_DIR / 'manual_review_200.csv'
SEED = 42
print('Data:', DATA_PATH)
print('Output:', OUTPUT_DIR)

Data: D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\hplt_en_uz_10k_clean.csv
Output: D:\dev\projects\fourlang_translation\results\data_quality\hplt_en_uz


In [2]:
df = pd.read_csv(DATA_PATH)
required = {'pair_id', 'en', 'uz', 'aligner_score', 'bicleaner_score', 'bifixer_score'}
missing = required - set(df.columns)
assert not missing, f'Missing columns: {sorted(missing)}'
for col in ['en', 'uz']:
    df[col] = df[col].fillna('').astype(str).str.strip()
for col in ['aligner_score', 'bicleaner_score', 'bifixer_score']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Rows:', len(df))
display(df[['aligner_score', 'bicleaner_score', 'bifixer_score']].describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]).T)

Rows: 10000


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
aligner_score,10000.0,0.645378,0.122493,0.500,0.500025,0.507117,0.517103,0.550481,0.615886,0.707107,0.816497,0.894427,1.000000,1.000000
bicleaner_score,10000.0,0.982706,0.034294,0.800,0.826000,0.906000,0.946000,0.984000,0.998000,1.000000,1.000000,1.000000,1.000000,1.000000
bifixer_score,10000.0,1.056220,0.681306,0.709,0.807798,0.897600,0.914390,0.933400,0.950100,0.969600,1.309630,1.693200,2.532023,45.939999


In [3]:
def normalized_text(text):
    return re.sub(r'[^\w]+', ' ', text.casefold(), flags=re.UNICODE).strip()

def numbers(text):
    return sorted(re.findall(r'\d+(?:[.,]\d+)?', text))

def has_word_repeat(text):
    return bool(re.search(r"\b([\w']+)\b(?:\s+\1\b){2,}", text.casefold(), flags=re.UNICODE))

df['en_chars_actual'] = df['en'].str.len()
df['uz_chars_actual'] = df['uz'].str.len()
shorter = df[['en_chars_actual', 'uz_chars_actual']].min(axis=1).clip(lower=1)
longer = df[['en_chars_actual', 'uz_chars_actual']].max(axis=1)
df['length_ratio'] = longer / shorter
df['flag_empty'] = (df['en_chars_actual'] == 0) | (df['uz_chars_actual'] == 0)
df['flag_identical'] = [normalized_text(a) == normalized_text(b) for a, b in zip(df['en'], df['uz'])]
df['flag_length_ratio'] = df['length_ratio'] > 2.0
df['flag_very_short'] = (df['en'].str.split().str.len() <= 2) | (df['uz'].str.split().str.len() <= 2)
df['flag_number_mismatch'] = [numbers(a) != numbers(b) for a, b in zip(df['en'], df['uz'])]
df['flag_web_noise'] = df['en'].str.contains(r'https?://|www\.|<[^>]+>|&(?:nbsp|amp|quot);', case=False, regex=True) | df['uz'].str.contains(r'https?://|www\.|<[^>]+>|&(?:nbsp|amp|quot);', case=False, regex=True)
df['flag_repeat'] = [has_word_repeat(a) or has_word_repeat(b) for a, b in zip(df['en'], df['uz'])]
df['flag_low_aligner'] = df['aligner_score'].fillna(-1) < 0.65
df['flag_low_bicleaner'] = df['bicleaner_score'].fillna(-1) < 0.95
df['flag_low_bifixer'] = df['bifixer_score'].fillna(-1) < 0.85
flag_columns = [c for c in df.columns if c.startswith('flag_')]
weights = {
    'flag_empty': 10, 'flag_identical': 6, 'flag_length_ratio': 3,
    'flag_very_short': 1, 'flag_number_mismatch': 2, 'flag_web_noise': 2,
    'flag_repeat': 2, 'flag_low_aligner': 4, 'flag_low_bicleaner': 3,
    'flag_low_bifixer': 2,
}
df['risk_score'] = sum(df[c].astype(int) * weights[c] for c in flag_columns)
df['risk_flags'] = df.apply(lambda row: '|'.join(c.removeprefix('flag_') for c in flag_columns if row[c]), axis=1)
summary = pd.DataFrame({
    'count': df[flag_columns].sum().astype(int),
    'percent': (df[flag_columns].mean() * 100).round(2),
}).sort_values('count', ascending=False)
display(summary)
print('Any automatic risk flag:', int((df.risk_score > 0).sum()), f"({(df.risk_score > 0).mean():.1%})")

,count,percent
flag_low_aligner,6064,60.64
flag_low_bicleaner,1116,11.16
flag_number_mismatch,531,5.31
flag_very_short,243,2.43
flag_low_bifixer,187,1.87
flag_length_ratio,177,1.77
flag_web_noise,13,0.13
flag_empty,0,0.00
flag_identical,0,0.00
flag_repeat,0,0.00


Any automatic risk flag: 6807 (68.1%)


In [4]:
risk_pool = df[df['risk_score'] > 0].sort_values(['risk_score', 'aligner_score'], ascending=[False, True])
risk_sample = risk_pool.head(min(100, len(risk_pool))).copy()
remaining = df[~df['pair_id'].isin(risk_sample['pair_id'])]
control_sample = remaining.sample(n=min(100, len(remaining)), random_state=SEED).copy()
risk_sample['sample_group'] = 'high_risk'
control_sample['sample_group'] = 'random_control'
review = pd.concat([risk_sample, control_sample], ignore_index=True)
review = review.sample(frac=1, random_state=SEED).reset_index(drop=True)
review.insert(0, 'review_id', np.arange(1, len(review) + 1))
review['manual_label'] = ''
review['manual_notes'] = ''
review_columns = [
    'review_id', 'pair_id', 'sample_group', 'en', 'uz',
    'aligner_score', 'bicleaner_score', 'bifixer_score',
    'length_ratio', 'risk_score', 'risk_flags',
    'manual_label', 'manual_notes',
]
review[review_columns].to_csv(REVIEW_PATH, index=False, encoding='utf-8-sig')
print('Saved:', REVIEW_PATH)
print('Fill manual_label with one of: correct, minor, wrong, junk')
display(review[review_columns].head(20))

Saved: D:\dev\projects\fourlang_translation\results\data_quality\hplt_en_uz\manual_review_200.csv
Fill manual_label with one of: correct, minor, wrong, junk


,review_id,pair_id,sample_group,en,uz,aligner_score,bicleaner_score,bifixer_score,length_ratio,risk_score,risk_flags,manual_label,manual_notes
0,1,hplt_00002652,high_risk,It's also pretty incredible that we've managed...,"Lekin, sizlardan 99%, ehtimol, biz uchun mosli...",0.538660,1.000,0.9426,2.521739,9,length_ratio|number_mismatch|low_aligner,,
1,2,hplt_00003570,high_risk,"Retrieved March 9, 2021.",Qaraldi: 8-mart 2021-yil.,0.516398,0.891,0.7861,1.041667,11,number_mismatch|low_aligner|low_bicleaner|low_...,,
2,3,hplt_00002549,high_risk,Preset support.,Oldindan o'rnatilgan qo'llab -quvvatlash .,0.577350,0.815,0.9616,2.800000,11,length_ratio|very_short|low_aligner|low_bicleaner,,
3,4,hplt_00005357,random_control,Unlike chat rouletteson other platforms you ca...,"Ruletka chatidan farqli o'laroq, siz tasodifiy...",0.517856,0.995,0.9682,1.054054,4,low_aligner,,
4,5,hplt_00007972,random_control,The journey will take about 5 minutes.,Safar taxminan 5 daqiqa davom etadi.,0.654976,0.976,0.9227,1.055556,0,,,
5,6,hplt_00002483,random_control,Another example: In case of increased body tem...,Yana bir misol: tana harorati ko'tarilgan taqd...,0.530595,1.000,0.9371,1.075145,4,low_aligner,,
6,7,hplt_00002750,high_risk,• double-effect HLM-1,Ichki sifatni nazorat qilish tartiblariquyidag...,0.512049,1.000,0.9888,3.333333,9,length_ratio|number_mismatch|low_aligner,,
7,8,hplt_00008992,random_control,The picture editor in Adobe Elements 14 is div...,Adobe Elements 14 -dagi rasm muharriri uchta r...,0.749269,0.999,0.9084,1.021277,0,,,
8,9,hplt_00003037,random_control,Send your suggestions and wishes to the mail,Sizning takliflaringizni va tilaklaringizni po...,0.882207,0.966,0.9888,1.386364,0,,,
9,10,hplt_00004553,high_risk,Tashkent branch,-issiqlik almashuvchisi - 1,0.547723,0.872,2.6136,1.800000,10,very_short|number_mismatch|low_aligner|low_bic...,,


In [5]:
columns_to_show = ['pair_id', 'en', 'uz', 'aligner_score', 'bicleaner_score', 'bifixer_score', 'length_ratio', 'risk_score', 'risk_flags']
display(df.sort_values(['risk_score', 'aligner_score'], ascending=[False, True])[columns_to_show].head(50))

,pair_id,en,uz,aligner_score,bicleaner_score,bifixer_score,length_ratio,risk_score,risk_flags
7177,hplt_00007177,Retrieved 17 May 2010.,Ibn Bajja.,0.633320,0.938,0.7647,2.200000,15,length_ratio|very_short|number_mismatch|low_al...
1338,hplt_00001338,High profit margin,Chatbox va yordamchi xodimlar 24/7 ishlaydi,0.500000,0.827,0.9146,2.388889,12,length_ratio|number_mismatch|low_aligner|low_b...
4651,hplt_00004651,As a result the core temperature of main seque...,Sintezlangan atom yadrosi sof massasi reagentl...,0.514517,0.905,0.9943,2.153846,12,length_ratio|number_mismatch|low_aligner|low_b...
6406,hplt_00006406,Spyware Loop.,Qaraldi: 13-sentabr 2012-yil.,0.526764,0.954,0.8165,2.230769,12,length_ratio|very_short|number_mismatch|low_al...
8942,hplt_00008942,DTIC Document.,Qaraldi: 13-sentabr 2012-yil.,0.526764,0.954,0.8165,2.071429,12,length_ratio|very_short|number_mismatch|low_al...
7175,hplt_00007175,Most stars are between 1 billion and 10 billio...,Yulduz yadrosidagi geliyni yoqib tugatganidan ...,0.583874,0.846,0.9101,2.017241,12,length_ratio|number_mismatch|low_aligner|low_b...
8878,hplt_00008878,Most stars are between 1 billion and 10 billio...,[tahrir] Magnit maydoni,0.583874,0.846,0.9101,2.521739,12,length_ratio|number_mismatch|low_aligner|low_b...
3225,hplt_00003225,The European Central Bank (ECB) is the central...,Yevropa markaziy banki (YeMB) — Yevropa valyut...,0.585261,0.922,1.4948,2.607595,12,length_ratio|number_mismatch|low_aligner|low_b...
5584,hplt_00005584,Yushchenko stated that he wants to continue to...,Oila va shaxsiy hayot[tahrir | manbasini tahri...,0.598083,0.912,0.9760,2.615385,12,length_ratio|number_mismatch|low_aligner|low_b...
8068,hplt_00008068,Spiegel Online.,Qaraldi: 10-yanvar 2009-yil.,0.633320,0.948,0.8076,1.866667,12,very_short|number_mismatch|low_aligner|low_bic...


In [6]:
labeled = pd.read_csv(REVIEW_PATH, keep_default_na=False)
allowed = {'', 'correct', 'minor', 'wrong', 'junk'}
unknown = set(labeled['manual_label'].str.strip().str.lower()) - allowed
assert not unknown, f'Unknown labels: {sorted(unknown)}'
labeled['manual_label'] = labeled['manual_label'].str.strip().str.lower()
completed = labeled[labeled['manual_label'] != ''].copy()
print(f'Labeled: {len(completed)}/{len(labeled)}')
if len(completed):
    display(pd.crosstab(completed['sample_group'], completed['manual_label'], margins=True))
    severe = completed['manual_label'].isin(['wrong', 'junk'])
    print(f'Wrong/junk among labeled: {severe.mean():.1%}')
    completed[completed['manual_label'].isin(['correct', 'minor'])][['pair_id']].to_csv(OUTPUT_DIR / 'review_keep_pair_ids.csv', index=False, encoding='utf-8-sig')
    completed[completed['manual_label'].isin(['wrong', 'junk'])][['pair_id']].to_csv(OUTPUT_DIR / 'review_reject_pair_ids.csv', index=False, encoding='utf-8-sig')
else:
    print('Open manual_review_200.csv, label the rows, save it, then rerun this cell.')

Labeled: 0/200
Open manual_review_200.csv, label the rows, save it, then rerun this cell.


In [7]:
audit_report = {
    'rows': int(len(df)),
    'automatic_flag_counts': {c.removeprefix('flag_'): int(df[c].sum()) for c in flag_columns},
    'automatic_flag_percent': {c.removeprefix('flag_'): round(float(df[c].mean() * 100), 3) for c in flag_columns},
    'any_flag_count': int((df['risk_score'] > 0).sum()),
    'review_file': str(REVIEW_PATH),
}
(OUTPUT_DIR / 'automatic_audit_report.json').write_text(json.dumps(audit_report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(audit_report, ensure_ascii=False, indent=2))

{
  "rows": 10000,
  "automatic_flag_counts": {
    "empty": 0,
    "identical": 0,
    "length_ratio": 177,
    "very_short": 243,
    "number_mismatch": 531,
    "web_noise": 13,
    "repeat": 0,
    "low_aligner": 6064,
    "low_bicleaner": 1116,
    "low_bifixer": 187
  },
  "automatic_flag_percent": {
    "empty": 0.0,
    "identical": 0.0,
    "length_ratio": 1.77,
    "very_short": 2.43,
    "number_mismatch": 5.31,
    "web_noise": 0.13,
    "repeat": 0.0,
    "low_aligner": 60.64,
    "low_bicleaner": 11.16,
    "low_bifixer": 1.87
  },
  "any_flag_count": 6807,
  "review_file": "D:\\dev\\projects\\fourlang_translation\\results\\data_quality\\hplt_en_uz\\manual_review_200.csv"
}
